# Taller práctico de analítica de datos
## Hábitos de vida y niveles de obesidad: ¿qué factores pesan más allá del propio peso?

**Herramientas:** Anaconda · Jupyter Notebook · Python (pandas, seaborn, scikit-learn)

**Dataset:** *Estimation of Obesity Levels Based on Eating Habits and Physical Condition*
(UCI Machine Learning Repository, ID 544) — 2 111 personas de Colombia, Perú y México, con
17 variables sobre alimentación, actividad física, transporte y datos demográficos, más una
etiqueta de nivel de obesidad (7 categorías, de *Peso insuficiente* a *Obesidad tipo III*).

---

### 🎯 Pregunta clave del taller

> **¿Qué hábitos de vida y factores sociodemográficos están más asociados con el nivel de
> obesidad de una persona, más allá del propio peso y la altura?**

Esta pregunta se responde en dos partes:

1. Primero comprobamos **cómo se construyó la etiqueta** del dataset (para evitar caer en una
   trampa muy común de la analítica de datos: la *fuga de información*, o *data leakage*).
2. Después entrenamos un modelo que **excluye deliberadamente el peso y la altura**, para
   descubrir qué hábitos (comer entre horas, actividad física, transporte, consumo de agua,
   antecedentes familiares, etc.) explican mejor el nivel de obesidad.

Este es el tipo de decisión que un analista de datos debe tomar constantemente: no basta con
maximizar la exactitud de un modelo, hay que preguntarse **qué pregunta de negocio o de salud
pública se está respondiendo realmente**.


## 0. Preparar el entorno con Anaconda (versión web)

Este taller está pensado para ejecutarse en **Anaconda web** (anaconda.cloud), importando
directamente este archivo `.ipynb`:

1. Entrar en [anaconda.cloud](https://anaconda.cloud) e iniciar sesión con tu cuenta de Anaconda.
2. Abrir un **notebook session** (entorno Jupyter en la nube, sin instalación local).
3. **Importar** este archivo `.ipynb` (botón *Upload* / *Import notebook*) para cargarlo en el
   entorno.
4. Ejecutar la celda de librerías de más abajo. Si alguna falta (por ejemplo `ucimlrepo`, que no
   siempre viene preinstalada), instalarla desde la propia celda con `%pip install ucimlrepo`.
5. A partir de ahí, ejecutar las celdas en orden, de arriba hacia abajo.

La primera celda de código importa las librerías que usaremos durante todo el taller.


In [ ]:
# Si el entorno de Anaconda web no trae alguna librería instalada, descomenta la línea:
!pip install ucimlrepo
!pip install --upgrade seaborn
# Librerías de trabajo

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Estilo de las gráficas
sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.dpi"] = 110

pd.set_option("display.max_columns", None)
print("Entorno listo ✔")


## 1. Descarga del dataset

Usamos el paquete oficial **`ucimlrepo`**, mantenido por el propio repositorio de la UCI, que
descarga los datos directamente desde su fuente original (`archive.ics.uci.edu`) sin necesidad de
guardar ninguna copia manual. Como alternativa (por ejemplo, si no hay acceso a ese dominio en tu
red), se incluye una descarga de respaldo desde una copia CSV de solo lectura del mismo dataset.


In [ ]:
# --- Descarga oficial vía UCI ML Repository ---
dataset_id = 544  # Estimation of obesity levels based on eating habits and physical condition

try:
    from ucimlrepo import fetch_ucirepo
    obesity = fetch_ucirepo(id=dataset_id)
    df_raw = pd.concat([obesity.data.features, obesity.data.targets], axis=1)
    print("Descarga vía ucimlrepo (UCI oficial) ✔")
except Exception as e:
    # --- Alternativa: copia CSV directa (mismo dataset, mismas 2111 filas) ---
    print(f"No se pudo usar ucimlrepo ({e}); usando descarga alternativa por CSV...")
    url = "https://raw.githubusercontent.com/SmilodonCub/DATA605/master/ObesityDataSet_raw_and_data_sinthetic.csv"
    df_raw = pd.read_csv(url)
    df_raw.to_csv("ObesityDataSet.csv", index=False)
    print("Descarga alternativa vía CSV ✔")

print("Dimensiones:", df_raw.shape)
df_raw.head()

**Diccionario rápido de columnas** (nombres originales del dataset):

| Columna | Significado |
|---|---|
| `Gender`, `Age`, `Height`, `Weight` | Datos demográficos y antropométricos |
| `family_history_with_overweight` | ¿Algún familiar ha tenido sobrepeso? |
| `FAVC` | Consumo frecuente de alimentos hipercalóricos |
| `FCVC` | Frecuencia de consumo de vegetales |
| `NCP` | Número de comidas principales al día |
| `CAEC` | Consumo de alimentos entre comidas |
| `SMOKE` | Fuma |
| `CH2O` | Consumo diario de agua |
| `SCC` | Monitorea las calorías que consume |
| `FAF` | Frecuencia de actividad física |
| `TUE` | Tiempo de uso de dispositivos electrónicos |
| `CALC` | Consumo de alcohol |
| `MTRANS` | Medio de transporte habitual |
| `NObeyesdad` | **Variable objetivo**: nivel de obesidad (7 categorías) |


## 2. Exploración inicial y limpieza de datos

Antes de modelar, revisamos: tipos de datos, valores nulos, filas duplicadas y valores
fuera de rango. Esta etapa es donde se detectan la mayoría de los problemas que después
arruinan un análisis.


In [ ]:
print("Tipos de datos:")
print(df_raw.dtypes)
print("\nValores nulos por columna:")
print(df_raw.isna().sum())
print("\nFilas duplicadas:", df_raw.duplicated().sum())


In [ ]:
# Quitamos duplicados exactos (mismas 17 respuestas repetidas)
df = df_raw.drop_duplicates().reset_index(drop=True)
print(f"Filas antes: {df_raw.shape[0]}  →  Filas después de limpiar duplicados: {df.shape[0]}")

# Revisión de rangos plausibles en variables numéricas clave
df[["Age", "Height", "Weight"]].describe()


Los rangos de edad (14–61 años), altura (1.45–1.98 m) y peso (39–173 kg) son plausibles
para una muestra de población general adulta/adolescente, así que no se eliminan outliers en
esta etapa. Ahora calculamos el **Índice de Masa Corporal (IMC)** para verificar cómo se
construyó la variable objetivo — este paso es clave para nuestra pregunta.


In [ ]:
df["BMI"] = df["Weight"] / (df["Height"] ** 2)

orden_clases = ["Insufficient_Weight", "Normal_Weight", "Overweight_Level_I", "Overweight_Level_II",
                 "Obesity_Type_I", "Obesity_Type_II", "Obesity_Type_III"]
etiquetas_es = ["Peso insuficiente", "Peso normal", "Sobrepeso I", "Sobrepeso II",
                "Obesidad I", "Obesidad II", "Obesidad III"]
mapa_etiquetas = dict(zip(orden_clases, etiquetas_es))

df.groupby("NObeyesdad")["BMI"].agg(["min", "max", "mean"]).reindex(orden_clases)


### ⚠️ Hallazgo importante: fuga de datos (*data leakage*)

Los rangos de IMC por categoría **no se solapan entre sí** y coinciden casi exactamente con los
umbrales clínicos estándar (bajo peso &lt; 18.5, normal 18.5–25, sobrepeso 25–30, obesidad &gt; 30,
etc.). Esto confirma que **`NObeyesdad` fue etiquetada directamente a partir del IMC**, es decir,
a partir de `Weight` y `Height`.

**Consecuencia práctica:** si entrenamos un modelo que use `Weight` y `Height` como variables de
entrada, el modelo no estará "descubriendo" nada — simplemente aprenderá a recalcular una fórmula
matemática (fuga de datos). Por eso, para responder a la verdadera pregunta del taller
(*qué hábitos se asocian al nivel de obesidad*), es necesario **excluir el peso y la altura del
modelo**. Lo demostramos a continuación con dos modelos comparativos.


## 3. Análisis exploratorio (EDA)

### 3.1 Distribución de la variable objetivo


In [ ]:
plt.figure(figsize=(8, 5))
conteo = df["NObeyesdad"].value_counts().reindex(orden_clases)
sns.barplot(x=[mapa_etiquetas[c] for c in orden_clases], y=conteo.values,
            hue=[mapa_etiquetas[c] for c in orden_clases], palette="viridis", legend=False)
plt.xticks(rotation=35, ha="right")
plt.ylabel("Número de personas")
plt.xlabel("")
plt.title("Distribución de niveles de obesidad en la muestra")
plt.tight_layout()
plt.show()


Las clases están razonablemente balanceadas (entre 272 y 351 personas por categoría), lo
que facilita el entrenamiento de un modelo de clasificación sin necesidad de técnicas especiales
de rebalanceo.

### 3.2 Confirmación visual del IMC por clase


In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x="NObeyesdad", y="BMI", data=df, order=orden_clases,
            hue="NObeyesdad", palette="viridis", legend=False)
plt.xticks(ticks=range(7), labels=etiquetas_es, rotation=35, ha="right")
plt.xlabel("")
plt.ylabel("IMC (kg/m²)")
plt.title("IMC calculado vs. categoría asignada\n(confirma el criterio de etiquetado)")
plt.tight_layout()
plt.show()


### 3.3 Correlación entre variables numéricas


In [ ]:
num_cols = ["Age", "Height", "Weight", "FCVC", "NCP", "CH2O", "FAF", "TUE", "BMI"]
plt.figure(figsize=(7, 6))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, cbar_kws={"shrink": .8})
plt.title("Correlación entre variables numéricas")
plt.tight_layout()
plt.show()


Como es esperable, `Weight` y `BMI` están muy correlacionados (0.9+). Entre las variables
de hábitos, ninguna tiene una correlación lineal fuerte por sí sola con el IMC — lo cual sugiere
que su efecto es más bien **conjunto y no lineal**, ideal para un modelo basado en árboles como
Random Forest.


## 4. Preparación de los datos para el modelo

Codificamos las variables categóricas:
- Variables binarias (`sí`/`no`) → 0/1.
- Variables ordinales (`CAEC`, `CALC`: nunca/a veces/frecuente/siempre) → 0-3.
- `Gender` → 0/1.
- `MTRANS` (medio de transporte, sin orden natural) → *one-hot encoding*.


In [ ]:
df_mod = df.copy()

# Binarias
for c in ["family_history_with_overweight", "FAVC", "SMOKE", "SCC"]:
    df_mod[c] = df_mod[c].map({"yes": 1, "no": 0})

# Ordinales
mapa_ordinal = {"no": 0, "Sometimes": 1, "Frequently": 2, "Always": 3}
df_mod["CAEC"] = df_mod["CAEC"].map(mapa_ordinal)
df_mod["CALC"] = df_mod["CALC"].map(mapa_ordinal)

# Género
df_mod["Gender"] = df_mod["Gender"].map({"Male": 1, "Female": 0})

# Transporte -> one-hot
df_mod = pd.get_dummies(df_mod, columns=["MTRANS"], prefix="MTRANS")

# Variable objetivo codificada respetando el orden clínico
le = LabelEncoder()
le.fit(orden_clases)
y = le.transform(df_mod["NObeyesdad"])

df_mod.head()


## 5. Modelo A — control: ¿qué pasa si dejamos peso y altura?

Entrenamos un `RandomForestClassifier` **incluyendo** `Weight` y `Height` como variables de
entrada. Esperamos una exactitud artificialmente alta, confirmando la fuga de datos detectada
en la sección 2.


In [ ]:
Xa = df_mod.drop(columns=["NObeyesdad", "BMI"])
Xa_train, Xa_test, y_train, y_test = train_test_split(
    Xa, y, test_size=0.2, random_state=42, stratify=y
)

modelo_A = RandomForestClassifier(n_estimators=300, random_state=42)
modelo_A.fit(Xa_train, y_train)
pred_A = modelo_A.predict(Xa_test)
acc_A = accuracy_score(y_test, pred_A)
print(f"Exactitud del Modelo A (incluye peso/altura): {acc_A:.1%}")


In [ ]:
cm_A = confusion_matrix(y_test, pred_A)
plt.figure(figsize=(7, 6))
sns.heatmap(cm_A, annot=True, fmt="d", cmap="Blues",
            xticklabels=etiquetas_es, yticklabels=etiquetas_es, cbar=False)
plt.xticks(rotation=40, ha="right")
plt.yticks(rotation=0)
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.title(f"Matriz de confusión — Modelo A (incluye peso/altura)\nExactitud = {acc_A:.1%}")
plt.tight_layout()
plt.show()


Como se sospechaba, la exactitud es muy alta (>95%) y los errores se concentran casi
exclusivamente entre categorías **adyacentes** (p. ej. Obesidad I vs. Obesidad II). El modelo no
está aprendiendo hábitos: está recalculando el IMC. **Este modelo no sirve para responder
nuestra pregunta** — pero es un ejercicio muy útil para aprender a detectar fuga de datos antes
de sacar conclusiones erróneas.


## 6. Modelo B — el modelo que responde la pregunta del taller

Ahora entrenamos el modelo **excluyendo** `Weight`, `Height` y `BMI`. Solo quedan variables de
**hábitos y demografía**: alimentación, actividad física, transporte, sueño/pantallas,
antecedentes familiares, edad y género.


In [ ]:
Xb = df_mod.drop(columns=["NObeyesdad", "BMI", "Weight", "Height"])
Xb_train, Xb_test, y_train2, y_test2 = train_test_split(
    Xb, y, test_size=0.2, random_state=42, stratify=y
)

modelo_B = RandomForestClassifier(n_estimators=400, max_depth=12, random_state=42)
modelo_B.fit(Xb_train, y_train2)
pred_B = modelo_B.predict(Xb_test)
acc_B = accuracy_score(y_test2, pred_B)
print(f"Exactitud del Modelo B (solo hábitos/demografía): {acc_B:.1%}")
print()
print(classification_report(y_test2, pred_B, target_names=etiquetas_es))


In [ ]:
cm_B = confusion_matrix(y_test2, pred_B)
plt.figure(figsize=(7, 6))
sns.heatmap(cm_B, annot=True, fmt="d", cmap="Purples",
            xticklabels=etiquetas_es, yticklabels=etiquetas_es, cbar=False)
plt.xticks(rotation=40, ha="right")
plt.yticks(rotation=0)
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.title(f"Matriz de confusión — Modelo B (solo hábitos/demografía)\nExactitud = {acc_B:.1%}")
plt.tight_layout()
plt.show()


Sin peso ni altura, el modelo **todavía alcanza ~84% de exactitud** solo a partir de
hábitos y datos demográficos — un resultado nada trivial (el azar entre 7 clases sería ~14%).
Esto confirma que los hábitos de vida sí contienen señal real y aprovechable para estimar el
nivel de obesidad de una persona.


## 7. ¿Qué hábitos pesan más? Importancia de variables


In [ ]:
importancias = pd.Series(modelo_B.feature_importances_, index=Xb.columns).sort_values(ascending=True).tail(12)

nombres_es = {
    "Age": "Edad", "FCVC": "Consumo de vegetales", "NCP": "N° comidas principales",
    "FAF": "Actividad física", "TUE": "Tiempo en pantallas", "CH2O": "Consumo de agua",
    "Gender": "Género", "CALC": "Consumo de alcohol",
    "family_history_with_overweight": "Antecedentes familiares",
    "CAEC": "Picar entre comidas", "FAVC": "Consumo alta cal.", "SMOKE": "Fuma",
    "SCC": "Monitorea calorías", "MTRANS_Public_Transportation": "Transporte público",
    "MTRANS_Walking": "Camina", "MTRANS_Automobile": "Automóvil",
    "MTRANS_Motorbike": "Motocicleta", "MTRANS_Bike": "Bicicleta",
}
importancias.index = [nombres_es.get(i, i) for i in importancias.index]

plt.figure(figsize=(8, 6))
plt.barh(importancias.index, importancias.values,
         color=sns.color_palette("viridis", len(importancias)))
plt.xlabel("Importancia relativa")
plt.title("Factores de hábitos/demografía más asociados\ncon el nivel de obesidad (Modelo B)")
plt.tight_layout()
plt.show()


## 8. Conclusiones

Retomando la pregunta clave del taller — **¿qué hábitos de vida y factores sociodemográficos
están más asociados con el nivel de obesidad de una persona, más allá del propio peso y la
altura?** — el análisis permite concluir:

1. **La etiqueta del dataset está definida por el IMC.** Un modelo que incluya peso y altura
   solo "redescubre" esa fórmula (Modelo A, ~96% de exactitud) y no aporta información útil
   sobre *causas* o *hábitos* asociados al sobrepeso: es un caso de manual de **fuga de datos**.

2. **Los hábitos de vida sí predicen razonablemente bien el nivel de obesidad por sí solos**
   (Modelo B, ~84% de exactitud sin usar peso ni altura), muy por encima del azar (~14% con 7
   clases).

3. Los factores más asociados, según la importancia de variables del Modelo B, son —en este
   orden aproximado—: **edad**, **frecuencia de consumo de vegetales**, **número de comidas
   principales al día**, **frecuencia de actividad física**, **tiempo frente a pantallas**,
   **consumo diario de agua**, **género** y **antecedentes familiares de sobrepeso**. El medio
   de transporte y el consumo de alcohol también aportan señal, aunque en menor medida.

4. **Implicación práctica:** una intervención de salud pública centrada en promover el consumo
   de vegetales, la actividad física regular, reducir el tiempo de pantalla y fomentar hábitos
   alimenticios estructurados (más comidas principales, menos picoteo) tiene, según estos datos,
   más potencial de asociarse a un menor nivel de obesidad que solo "vigilar el peso".

**Limitaciones a tener en cuenta:** el dataset combina datos reales de encuesta (23%) con datos
sintéticos generados con SMOTE (77%) para balancear las clases, y la muestra proviene solo de
tres países (Colombia, Perú y México). Las conclusiones deben interpretarse como **asociaciones
dentro de esta muestra**, no como relaciones causales generalizables a cualquier población.

---

### Ideas para extender el taller

- Probar otros modelos (regresión logística, XGBoost) y comparar su exactitud e interpretabilidad.
- Aplicar `SHAP` para explicar predicciones individuales, no solo la importancia global.
- Convertir el problema en regresión directa sobre el IMC en lugar de clasificación por categorías.
- Analizar interacciones entre variables (p. ej. actividad física × antecedentes familiares).

### Referencias

- Palechor, F. M., & de la Hoz Manotas, A. (2019). *Dataset for estimation of obesity levels
  based on eating habits and physical condition in individuals from Colombia, Peru and Mexico*.
  UCI Machine Learning Repository. https://doi.org/10.24432/C5H31Z
